## 1. Library Imports

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from pandas.plotting import scatter_matrix

## 2. Data Loading

In [2]:
# training = pd.read_parquet("data/train_laps.parquet", filters=[("race_id", "==", "2023_01")]) 

training = pd.read_parquet("train_laps.parquet")
test = pd.read_parquet("test_laps.parquet")  

## 3. EDA

In [3]:
# Important info (data description from kaggle): 

# Main Dataset
# The main dataset is in training_laps.parquet and test_laps.parquet. 
# They were generated from data on Formula 1 races 2018-2026: 95 training races, 92 test races. 
# The test races are incomplete : Each test race is cut at a cutoff lap. 
# The cutoff lap was selected when the leader finished a lap between 30 % and 50 % of the race (uniformly at random). 
# It used the column "race_time_s"; you can use this too to generate cuts. 
# Every car shows only the laps it had finished by then, plus the oracle columns. 
# You will be scored on the missing part.

# Oracle columns
# Some future events may influence tyre strategy. 
# Teams often make strategies for different scenarios. 
# For you to be able to use at least a little bit of this information, 
# we have added some "look-ahead" columns that let you know the future: safety car in n+14 laps, for example. 
# Or rain. 
# These can be useful for wet races, amongst other things.

### 3.1 Quick Look (First look to get an idea)

In [4]:
# Info on the training data
training.info() 
# the 'next_compound' column here (column n°17) is the one we want to predict. It's not present in the test data

<class 'pandas.DataFrame'>
RangeIndex: 104757 entries, 0 to 104756
Data columns (total 72 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   lap_id                           104757 non-null  str    
 1   race_id                          104757 non-null  str    
 2   year                             104757 non-null  int64  
 3   round                            104757 non-null  int64  
 4   race_name                        104757 non-null  str    
 5   circuit                          104757 non-null  str    
 6   circuit_id                       104757 non-null  str    
 7   date                             104757 non-null  object 
 8   driver_code                      104757 non-null  str    
 9   driver_number                    104757 non-null  Int64  
 10  team                             104754 non-null  str    
 11  circuit_length_km                104757 non-null  float64
 12  turns        

In [5]:
training.isna().sum()

## Comments:
## - There's a lot of missing data. 
## - Ask TA if we can just use outside data to fill it 
##  (since this is considered cheating : "Using data beyond that provided in this competition. 
##  Notably, looking up the data of races in the test sets to improve your score in any way")
##  Update: he said no about outside data, but we're allowed to use the given data to fill in the missing values 
## - The goal is to fill as much missing values as possible using the data we have (this applies for both traning and test data)
## - The column 'next_compound' (the one we want to predict) has 3566 missing values (!!!!!!!!)

lap_id                       0
race_id                      0
year                         0
round                        0
race_name                    0
                         ...  
virtual_sc_Lplus14       22536
red_flag_Lplus14         22536
rainfall_Lplus14         22536
wind_speed_Lplus14_ms    22536
pressure_Lplus14_mbar    22536
Length: 72, dtype: int64

I tried to do some plotting to have an idea about the correlation between the data (I tried running the scatter_matrix function and training.corr(), but I got errors).
- I got an error when running training.corr() because of the column format (the column date has is a datetime.date object column, this was mainly the issue behind the error)
- For scatter_matrix(), the issue is the boolean columns.

I used claude to compute the correlation between variables whatever their type is, and especially the correlation with the 'next_compound' column (we have to bear in mind that the missing values haven't been dealt with yet)

In [6]:
def cramers_v(x, y):
    d = pd.DataFrame({"x": x, "y": y}).dropna()
    if d["x"].nunique() < 2 or d["y"].nunique() < 2:
        return np.nan
    ct = pd.crosstab(d["x"], d["y"])
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return np.sqrt(phi2 / min(kc - 1, rc - 1)) if min(kc - 1, rc - 1) > 0 else np.nan

def corr_ratio(cat, num):
    d = pd.DataFrame({"c": cat, "n": num}).dropna()
    if d["c"].nunique() < 2:
        return np.nan
    g = d.groupby("c")["n"].agg(["count", "mean"])
    ss_between = (g["count"] * (g["mean"] - d["n"].mean()) ** 2).sum()
    ss_total = ((d["n"] - d["n"].mean()) ** 2).sum()
    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan

def association_matrix(data, cat_cols, num_cols):
    cols = list(num_cols) + list(cat_cols)
    out = pd.DataFrame(np.nan, index=cols, columns=cols)
    for i, a in enumerate(cols):
        for b in cols[i:]:
            a_cat, b_cat = a in cat_cols, b in cat_cols
            if a == b:          v = 1.0
            elif a_cat and b_cat: v = cramers_v(data[a], data[b])
            elif a_cat:         v = corr_ratio(data[a], data[b])
            elif b_cat:         v = corr_ratio(data[b], data[a])
            else:               v = data[a].corr(data[b])
            out.loc[a, b] = out.loc[b, a] = v
    return out

# columns to leave out (identifiers / duplicates)
drop = ["lap_id", "date", "race_id", "circuit_id", "result_status"]

cat_cols = [c for c in training.select_dtypes(exclude=["number", "boolean", "bool"]).columns]
num_cols = list(training.select_dtypes(include=["number", "boolean", "bool"]).columns)

data = training.copy()
for c in num_cols: data[c] = data[c].astype("float64")   # fixes the pd.NA errors
for c in cat_cols: data[c] = data[c].astype("object")

data = data.sample(15000, random_state=0)                # speed: the full set is slow
am = association_matrix(data, cat_cols, num_cols)

# 1) What matters most for the target (according to Claude)
print(am["next_compound"].drop("next_compound").abs().sort_values(ascending=False).head(20)) 
## We only get 20 correlations. I ran the same command without .head(20) to get all the correlations in the next cell

'''
# 2) Full heatmap
plt.figure(figsize=(22, 18))
sns.heatmap(am, cmap="viridis", vmin=0, vmax=1)
plt.show()
'''

laps_remaining        0.727841
lap_number            0.721399
race_time_s           0.719492
stint                 0.557815
date                  0.423342
race_id               0.423342
compound              0.341231
race_name             0.302117
circuit_id            0.299643
circuit               0.299643
rainfall_Lplus14      0.267800
laps_down             0.261454
humidity_pct          0.240177
gap_to_leader_s       0.234776
tyre_life_laps        0.230605
track_temp_c          0.217161
air_temp_c            0.198830
pit_lane_time_s       0.191925
rainfall              0.188005
safety_car_Lplus14    0.178549
Name: next_compound, dtype: float64


'\n# 2) Full heatmap\nplt.figure(figsize=(22, 18))\nsns.heatmap(am, cmap="viridis", vmin=0, vmax=1)\nplt.show()\n'

In [7]:
# Correlation between the 'next_compound' column and the other columns
pre_missingvalues_filling = am["next_compound"].drop("next_compound").abs().sort_values(ascending=False)
pre_missingvalues_filling
## WE STILL HAVEN'T TOUCHED THE MISSING VALUES
## So the values might sometimes not make sense

laps_remaining           0.727841
lap_number               0.721399
race_time_s              0.719492
stint                    0.557815
date                     0.423342
                           ...   
drive_through_pending    0.018719
black_orange_flag        0.018616
lap_time_s               0.018354
stop_go_pending          0.014388
lap_id                        NaN
Name: next_compound, Length: 71, dtype: float64

In [8]:
# Info on the test data
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 40901 entries, 0 to 40900
Data columns (total 71 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   lap_id                           40901 non-null  str    
 1   race_id                          40901 non-null  str    
 2   year                             40901 non-null  int64  
 3   round                            40901 non-null  int64  
 4   race_name                        40901 non-null  str    
 5   circuit                          40901 non-null  str    
 6   circuit_id                       40901 non-null  str    
 7   date                             40901 non-null  object 
 8   driver_code                      40901 non-null  str    
 9   driver_number                    40901 non-null  Int64  
 10  team                             40900 non-null  str    
 11  circuit_length_km                40901 non-null  float64
 12  turns                        

In [9]:
test.isna().sum()
# We get the same result if we run the next 2 lines
#test.isnull().any() # returns a boolean if the column contains null variables or not (True = contains & False = does not contain)
#test.isnull().sum()

## Comments:
## - Here there's only 1 missing value in the column "team". 
## - And 1 missing value in the 'driving_time_s'
## 

lap_id                     0
race_id                    0
year                       0
round                      0
race_name                  0
                        ... 
virtual_sc_Lplus14       181
red_flag_Lplus14         181
rainfall_Lplus14         181
wind_speed_Lplus14_ms    181
pressure_Lplus14_mbar    181
Length: 71, dtype: int64

### 3.2 Dealing with missing data

#### 3.2.1 Training Dataset

##### Missing teams

COLUMN : team

DESCRIPTION : Team name as used that season.

In [10]:
# There's 3 missing values in the column "team". 
# By figuring out who's the driver and which year was it, we can fill out those values (no outside data needed, just some data manipulation)

training[training['team'].isna()] # 3 missing values

## - Alonso (2018, Race 18, 5th lap)
## - Grosjean (2018, Race 18, 12th lap)
## - Ricciardo (2018, Race 18, 49th lap)


# The goal now is to figure out which team each one of those drivers belonged to during that race, 
# and use this to fill the missing value

training[(training['race_id'] == '2018_18') & (training['driver_code'].isin(['ALO','GRO', 'RIC']))]

## Alonso was at McLaren,
## GrosJean was at Haas F1 Team,
## and Ricciardo was at Red Bull Racing


# Fill the missing values with the actual teams

replacements = pd.Series(['McLaren', 'Haas F1 Team', 'Red Bull Racing'], index=[8147, 8316, 8748]) 
## Those are the row indexes to make sure there's no mix up when it comes to puting what team where
training['team'] = training['team'].fillna(replacements)

# Just to check that the values where indeed filled 
training[training['team'].isna()] # If there's nothing displayed, then they're filled

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar


This was the easiest one to fill, since 
1. we don't have to take in consideration 1000 things at the same time
2. a driver cannot switch teams mid race 

All that we had to do is figure out, during the specific race in which the value was missing, what was the driver's team. 

---

##### Missing future compound

COLUMN : next_compound

DESCRIPTION : Target (train only), as if this lap L were the car's last visible lap: the tyre the car drives after its next pit stop after lap  L (a stop on lap L itself is already visible and does not count). SOFT / MEDIUM / HARD = softest / middle / hardest of the three dry compounds brought to that race; WET_OR_INTER = intermediate or full wet tyre; NONE = no further stop, the car drives to the flag. NA if the car retires without another stop, or the tyre after the stop is unknown.

In [11]:
training[training['next_compound'].isna()] # There's 3566 missing values here

# I have a theory that theyre missing because the drivers in question didn't finish the race 
# (race not finished = no pit stop)

# Checking the theory
training.loc[training['next_compound'].isna(), 'result_status'].value_counts()
# Theory confirmed

# And since the column's description explicitly mentions that the values are NA 
# if the car retires without another stop, or the tyre after the stop is unknown; 
# that's how we'll deal with the missing values. 
# We'll leave them as missing values (if we fill them with NA as a value, it's as if we created a new compound)

result_status
Retired             1376
Collision damage     330
Disqualified         330
Collision            271
Engine               224
Hydraulics           141
Power Unit            97
Accident              92
Brakes                80
Wheel                 73
Power loss            65
Gearbox               60
Exhaust               53
Suspension            53
Oil leak              32
Turbo                 32
Wheel nut             30
Spun off              29
Overheating           25
Puncture              24
Vibrations            17
Mechanical            16
Front wing            16
Water leak            15
Undertray             14
Withdrew              14
Fuel pressure         12
Rear wing             11
Battery               10
Transmission           9
Steering               6
Did not start          6
Differential           2
Driveshaft             1
Name: count, dtype: int64

---

In [104]:
training['next_compound'].unique()

<ArrowStringArray>
['HARD', 'NONE', 'MEDIUM', <NA>, 'SOFT', 'WET_OR_INTER']
Length: 6, dtype: string

##### Missing lap time

COLUMN : lap_time_s

DESCRIPTION : Time from crossing the timing line to crossing it again.

In [12]:
training[training['lap_time_s'].isna()] # There's 205 missing values here
# Same theory: unfinished race
# Subplot: The lap in question (where the lap time is missing) is the lap where the drivers faced the issue
# that made them stop the race (retire from the race due to whatever reason)

training.loc[training['lap_time_s'].isna(), 'result_status'].value_counts() 
# 202 who have a missing value here didn't finish the race for x reason
# The 3 others finished 1 lap behind

# Checking who are those 3 drivers
training[(training['lap_time_s'].isna()) & (training['result_status'] == '+1 Lap')] 

# After some consideration, we prefer leaving the 205 missing values as is. 
# Since it's a column with timings, It's not something we want to mess up.
# We'll still do some feature engineering/encoding.

# The encoding in question
# flag for the laps that have no lap timing at all -> keeps the information for the model
training["untimed_lap"] = training["lap_time_s"].isna()
print("untimed laps:", int(training["untimed_lap"].sum()))


untimed laps: 205


---

##### Missing driving time 

COLUMN : driving_time_s

DESCRIPTION : Time the car was actually driving on this lap.

In [13]:
training[training['driving_time_s'].isna()] # 3 missing values

# The 3 cases where there isn't a driving_time_s is because the drivers in question haven't finished the race: 
# Alonso had a collision in lap 5 
# GrosJean had a collision damage in lap 12
# Ricciardo had battery issues in lap 49

# There's also missing values in the 'next_compound column (the one we want to predict), 
# which makes some sense since they DNF'd, 
# so there's no pit stop.

# DECISION:
# I won't fill them because I think it might be complicated to compute it since we don't know at which point of the lap they had their issues.
# + it's a column with timings, so I also don't want to mess up

# But we'll also encode it (tant que je peux faire quelque chose qui sera utile pour que le modèle apprenne)

training["untimed_driving_time"] = training["driving_time_s"].isna()
print("untimed driving time:", int(training["untimed_driving_time"].sum()))

training[training['driving_time_s'].isna()] # It worked


untimed driving time: 3


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time
8147,2018_18_ALO_005,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ALO,14,...,89.0,100.133,False,False,False,False,1.0,1005.8,False,True
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,121.0,99.500,False,False,False,False,1.4,1005.8,False,True
8748,2018_18_RIC_049,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,107.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,False,True


In [14]:
# I had the intuition to check the laps of those 3 drivers during this particular race

training[(training['race_id'] == '2018_18') & (training['driver_code'] == 'ALO')] 
## For some reason, Alonso has only 2 recorded laps in this race
## And those laps are 1 & 5 (not even 1 & 2)

training[(training['race_id'] == '2018_18') & (training['driver_code'] == 'GRO')]
## Same thing here with Grosjean
## Only 3 recorded laps in this race
## And those laps are 1, 2 & 12

training[(training['race_id'] == '2018_18') & (training['driver_code'] == 'RIC')]
## Same thing here 
## Only 10 recorded laps in this race
## And those laps are from 1 to 9 & lap 49 

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time
8739,2018_18_RIC_001,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,83.0,98.992,False,False,False,False,1.4,1005.9,False,False
8740,2018_18_RIC_002,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,89.0,100.005,False,False,False,False,1.4,1005.9,False,False
8741,2018_18_RIC_003,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,121.0,99.852,False,False,False,False,2.7,1005.9,False,False
8742,2018_18_RIC_004,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,99.0,100.284,False,False,False,False,1.0,1005.8,False,False
8743,2018_18_RIC_005,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,82.0,100.133,False,False,False,False,1.0,1005.8,False,False
8744,2018_18_RIC_006,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,87.0,100.524,False,False,False,False,1.0,1005.8,False,False
8745,2018_18_RIC_007,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,89.0,100.525,False,False,False,False,1.0,1005.8,False,False
8746,2018_18_RIC_008,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,99.0,99.584,False,False,False,False,1.5,1005.8,False,False
8747,2018_18_RIC_009,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,100.0,98.856,False,False,False,False,1.9,1005.8,True,False
8748,2018_18_RIC_049,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,107.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,False,True


---

##### Missing sector 1

COLUMN : sector1_s

DESCRIPTION : Time for the first of three track sections.

In [15]:
training[training['sector1_s'].isna()] # 2322 missing values

# (I think I'll leave this one untouched, because since this is a column related to timing, I don't want to mess it up. 
# Add to that that we don't know what's the sector en question, so not to bullshit anything)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,307.0,88.248,False,False,False,False,4.1,997.0,False,False
58,2018_01_BOT_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,BOT,77,...,307.0,88.248,False,False,False,False,4.1,997.0,False,False
116,2018_01_ERI_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,307.0,88.248,False,False,False,False,4.1,997.0,False,False
122,2018_01_GAS_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,307.0,88.248,False,False,False,False,4.1,997.0,False,False
136,2018_01_GRO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GRO,8,...,307.0,88.248,False,False,False,False,4.1,997.0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104646,2026_15_SAI_001,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,SAI,55,...,63.0,107.810,False,False,False,False,0.3,1023.1,False,False
104697,2026_15_STR_001,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,63.0,107.810,False,False,False,False,0.3,1023.1,False,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,0.0,107.191,False,False,False,False,0.7,1023.0,True,False
104705,2026_15_STR_009,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,22.0,107.109,False,False,False,False,1.3,1022.9,True,False


##### Missing sector 2

COLUMN : sector2_s

DESCRIPTION : Time for the second section.

In [16]:
# Same thing applies here
training[training['sector2_s'].isna()] # 246 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,267.0,88.488,False,False,False,False,2.5,997.2,True,False
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,True,False
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,227.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,True,False
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,80.0,106.492,False,False,False,False,2.8,1022.0,True,False
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,92.0,106.722,False,False,False,False,2.0,1021.8,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,249.0,106.520,False,False,False,False,0.9,1023.0,True,False
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,348.0,105.005,False,False,False,False,0.0,1023.1,True,False
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,249.0,106.520,False,False,False,False,0.9,1023.0,True,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,0.0,107.191,False,False,False,False,0.7,1023.0,True,False


##### Missing sector 3

COLUMN : sector3_s

DESCRIPTION : Time for the third section.

In [17]:
# And here
training[training['sector3_s'].isna()] # 381

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,295.0,88.245,False,False,False,False,3.5,997.0,True,False
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,278.0,121.319,True,True,False,False,2.2,996.9,True,False
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,267.0,88.488,False,False,False,False,2.5,997.2,True,False
907,2018_01_VET_026,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,0.0,86.880,False,False,False,False,2.5,996.9,False,False
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,82.0,NaN,<NA>,<NA>,<NA>,<NA>,NaN,NaN,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,249.0,106.520,False,False,False,False,0.9,1023.0,True,False
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,348.0,105.005,False,False,False,False,0.0,1023.1,True,False
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,249.0,106.520,False,False,False,False,0.9,1023.0,True,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,0.0,107.191,False,False,False,False,0.7,1023.0,True,False


Here's what we'll do for the missing values for those 3 sectors (Used claude for that)



**What happened:** I tested whether `sector1 + sector2 + sector3 = lap_time` on all rows where all four values exist. It holds on
essentially every row (check the printed share). That means that when the lap time exists and **exactly one** sector is missing,
the missing one is simply `lap_time − (the other two)`.

The big sector 1 gap is mostly **lap 1** (standing start: the first sector is not timed the same way), but the formula still holds there.

**Decision:** rebuild a sector when exactly one is missing and the lap time exists. If two or more are missing, or the lap has no time, we leave `NaN` (unrecoverable), and we encode them (encoding them is my idea).

In [18]:
S = ["sector1_s", "sector2_s", "sector3_s"]

full = training[S + ["lap_time_s"]].notna().all(axis=1)
err = (training.loc[full, S].sum(axis=1) - training.loc[full, "lap_time_s"]).abs()
print(f"sector1+2+3 == lap_time (±0.01 s) on {len(err):,} complete rows: {(err < 0.01).mean():.4%}")

before = training[S].isna().sum()
one_missing = (training[S].isna().sum(axis=1) == 1) & training["lap_time_s"].notna()
for s in S:
    others = [c for c in S if c != s]
    m = one_missing & training[s].isna()
    est = training.loc[m, "lap_time_s"] - training.loc[m, others].sum(axis=1)
    est = est[est > 0]                               # safety: a sector time must be positive
    training.loc[est.index, s] = est

print(pd.DataFrame({"# of missing values before": before, "after": training[S].isna().sum()}))

sector1+2+3 == lap_time (±0.01 s) on 102,290 complete rows: 99.9296%
           # of missing values before  after
sector1_s                        2322    248
sector2_s                         246    245
sector3_s                         381    272


In [19]:
# We'll encode each column (just to be sure that the encoding is correct par rapport à each column)

## sector1_s
training["missing_sector1_time"] = training["sector1_s"].isna()
print("missing sector 1 time:", int(training["missing_sector1_time"].sum()))

## sector2_s
training["missing_sector2_time"] = training["sector2_s"].isna()
print("missing sector 2 time:", int(training["missing_sector2_time"].sum()))

## sector3_s
training["missing_sector3_time"] = training["sector3_s"].isna()
print("missing sector 3 time:", int(training["missing_sector3_time"].sum()))

training[training['sector1_s'].isna()] # It worked

missing sector 1 time: 248
missing sector 2 time: 245
missing sector 3 time: 272


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,False,2.8,1022.0,True,False,True,True,True
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,False,False,2.0,1021.8,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


---

##### Missing speed i1

COLUMN : speed_i1_kmh

DESCRIPTION: Speed at intermediate measuring point 1.

In [20]:
training[training['speed_i1_kmh'].isna()] # 565 missing values

# (I think I'll leave this one untouched, because since this is a column related to timing, 
# I don't want to mess it up.)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,False,2.8,1022.0,True,False,True,True,True
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,False,False,2.0,1021.8,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


##### Missing speed i2

COLUMN : speed_i2_kmh

DESCRIPTION : Speed at intermediate measuring point 2

In [21]:
training[training['speed_i2_kmh'].isna()] # 199 missing values
# Same thing here

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,False,2.8,1022.0,True,False,True,True,True
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,False,False,2.0,1021.8,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


##### Missing speed finish line

COLUMN : speed_finish_line_kmh

DESCRIPTION : Speed at the finish line

In [22]:
# And here
training[training['speed_finish_line_kmh'].isna()] # 3672 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
25,2018_01_ALO_026,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,2.5,996.9,False,False,False,False,False
82,2018_01_BOT_025,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,BOT,77,...,False,False,False,3.7,996.9,False,False,False,False,False
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,False,3.5,997.0,True,False,False,False,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,True,False,False,2.2,996.9,True,False,False,False,True
159,2018_01_GRO_024,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GRO,8,...,False,False,False,2.2,996.8,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104681,2026_15_SAI_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,SAI,55,...,False,False,False,0.0,1023.1,False,False,False,False,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True
104705,2026_15_STR_009,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,1.3,1022.9,True,False,True,True,True
104736,2026_15_VER_031,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,False,False,0.0,1022.9,False,False,False,False,False


##### Missing speed trap

COLUMN : speed_trap_kmh

DESCRIPTION : Speed at the speed trap (end of the longest straight)

In [23]:
# And here
training[training['speed_trap_kmh'].isna()] # 483 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,False,2.8,1022.0,True,False,True,True,True
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,False,False,2.0,1021.8,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


Here's what we'll do for the missing values for those speed parts (also used claude for that)


**What happened** (printed below): no race is completely missing any of them; the gaps are scattered.
* `speed_finish_line_kmh`: **3350** of the 3672 are laps with a pit stop (the car drives through the pit lane, not over the line at speed) + the 206 untimed laps → structural.
* Part of every other speed column is lap 1 (standing start) or untimed laps → structural.
* The rest are isolated sensor gaps (e.g. `2019_05` and `2019_06` have 135 missing `speed_i1_kmh` each). A speed depends on slipstream, fuel and DRS, so an estimate from other laps would be **invented data** that looks like a measurement.

**Decision:** leave them empty. They carry real information in their *pattern*: missing finish-line speed ≈ “pit stop lap”, which is useful for predicting the next tyre!
`model_ready()` turns them into `-1` for models that cannot handle `NaN`.

*The function `model_ready()` is at the end with its description*

In [24]:
# I didn't fill anything

---

##### Missing positions

COLUMN: position

DESCRIPTION : Running position at the end of the lap, 1 = leading.

In [25]:
training[training['position'].isna()] # 175 missing values

# I couldn't get a lot of details, 
# so what I did is that I looked at the reasons why the drivers didn't finish the race 
# (I looked at the result status)
# The code in the next cell allowed me to do that

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,False,2.8,1022.0,True,False,True,True,True
1400,2018_04_OCO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,OCO,31,...,False,False,False,2.0,1021.8,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


In [26]:
training.loc[training['position'].isna(), 'result_status'].value_counts()
# This is the result_status for those that have missing values in the 'position' column.

# Turns out that 3 of them have 'result_status' == +1 Lap, so I'll see if I can just figure out their position
# The 172 others had issues (the coresponding 'result_status' column had values like Engine, Retired...), 
# so it makes sense that there's no values in the 'position' column (race not finished = no position). 
# We can deal with it by putting DNF, DNS, ... as values in the 'position' column.

result_status
Retired             71
Collision           42
Accident            14
Engine              10
Power Unit           5
Did not start        4
Brakes               3
Hydraulics           3
Suspension           3
Spun off             3
+1 Lap               3
Puncture             2
Exhaust              2
Gearbox              2
Battery              2
Oil leak             1
Fuel pressure        1
Power loss           1
Collision damage     1
Transmission         1
Driveshaft           1
Name: count, dtype: int64

In [27]:
training[(training['position'].isna()) & (training['result_status'] == '+1 Lap')]

# So the 3 drivers with no position and a result status of +1 Lap are:
# Sergio Perez during the Japanese GP in 2019
# Nico Hulkenberg and Kevin Magnussen during the Italian GP in 2023


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
20128,2019_17_PER_052,2019_17,2019,17,Japanese Grand Prix,Suzuka Circuit,suzuka,2019-10-13,PER,11,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
62566,2023_14_HUL_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,HUL,27,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
62719,2023_14_MAG_051,2023_14,2023,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2023-09-03,MAG,20,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True


In [28]:
# Decision
# I'll leave this column as is to not mess up.
# We could try to teach the model to look up for certain stuff before predicting
# If anyone has an idea to how to deal with this column be my guest


In [29]:
# Check this duplicate issue for all positions
for i in range(1, 23):
    dups = training[training["position"] == i].groupby(["race_id", "lap_number", 'position']).size()
    print(f"Position : {i}")
    print(dups[dups > 1])

Position : 1
race_id  lap_number  position
2020_09  8           1           2
2021_05  30          1           2
2024_12  1           1           2
dtype: int64
Position : 2
race_id  lap_number  position
2023_15  1           2           2
2023_20  1           2           2
2024_04  1           2           2
2024_08  1           2           2
dtype: int64
Position : 3
race_id  lap_number  position
2023_20  1           3           2
2024_04  1           3           2
2024_08  1           3           2
dtype: int64
Position : 4
race_id  lap_number  position
2023_20  1           4           2
2024_08  1           4           2
dtype: int64
Position : 5
race_id  lap_number  position
2024_08  1           5           2
dtype: int64
Position : 6
race_id  lap_number  position
2026_06  44          6           2
dtype: int64
Position : 7
Series([], dtype: int64)
Position : 8
race_id  lap_number  position
2023_08  54          8           2
dtype: int64
Position : 9
Series([], dtype: int64)
Positio

There's a few positions that have duplicates

I used again Claude to deal with both missing values and duplicates. So here what it did

**What happened**
* 172 missing values are retirements, 3 are the `+1 Lap` finishers (PER 2019 Japan, HUL & MAG 2023 Italy).
* While checking `gap_to_leader_s` you found three rows with `position == 1` and no gap, and then more duplicates for other positions.
  The cause is the same everywhere: **a car with no race time (retired / not timed that lap) still carries a leftover `position`**, which collides with the
  real car in that place. In total, 33 untimed rows have a position, and 24 of them duplicate a timed driver's position (e.g. `2020_09_OCO_008`,
  `2021_05_BOT_030`, `2024_12_GAS_001` are *not* leaders: the real leader on that lap has `gap_to_leader_s = 0`).
* A separate, smaller problem: on about two dozen laps the position order differs from the order of `race_time_s` by one swap (e.g. HAM/VET in `2022_19` lap 40).
  That is **not missing data** (both values exist), so I do not change it, I only list it.

**Decisions**
1. **Retired cars, untimed lap** → `position = NaN` (they have no running position; this removes the false positions and the duplicates).
2. **The 3 `+1 Lap` finishers** → position of their previous lap (they finished, so they have a final position): PER 9, HUL 17, MAG 18.
   (On the PER lap `HUL` is also listed 9th because the file ranks *only timed cars* — that one duplicate is kept on purpose.)

In [30]:
KEY = ["race_id", "driver_code"]     # one driver in one race

# Same definition of "finished" as the one we agreed on:
# Finished, Lapped, and every "+N Lap(s)" EXCEPT "+15 Laps" (that driver is not a real finisher)
status = training["result_status"].astype("object")
finished = (status.isin(["Finished", "Lapped"]) | status.str.match(r"^\+\d+ Laps?$")) & (status != "+15 Laps")

def dup_positions(df):
    p = df.dropna(subset=["position"])
    g = p.groupby(["race_id", "lap_number", "position"]).size()
    return g[g > 1]

print("- Duplicate (race, lap, position) groups BEFORE:", len(dup_positions(training)))

untimed = training["race_time_s"].isna()

# "previous lap" of the same driver in the same race (only if it is really lap-1)
d = training.sort_values(KEY + ["lap_number"])
grp = d.groupby(KEY)
consecutive = ((d["lap_number"] - grp["lap_number"].shift(1)) == 1).reindex(training.index)
prev_position = grp["position"].shift(1).reindex(training.index)
prev_laps_down = grp["laps_down"].shift(1).reindex(training.index)

# 1) finishers whose last lap was not timed: keep their last known position / laps_down
m_fin = untimed & finished & consecutive
fill_pos = m_fin & training["position"].isna()
training.loc[fill_pos, "position"] = prev_position[fill_pos]
fill_ld = m_fin & training["laps_down"].isna()
training.loc[fill_ld, "laps_down"] = prev_laps_down[fill_ld]
print(training.loc[fill_pos, ["lap_id", "position", "laps_down", "result_status"]])

# 2) retired cars on an untimed lap: no running position
m_ret = untimed & ~finished
print("retired untimed rows that still had a position (set to NaN):", int((m_ret & training["position"].notna()).sum()))
training.loc[m_ret, "position"] = pd.NA

print("- Duplicate groups AFTER:", len(dup_positions(training)))
print(dup_positions(training))

- Duplicate (race, lap, position) groups BEFORE: 24
                lap_id  position  laps_down result_status
20128  2019_17_PER_052         9          1        +1 Lap
62566  2023_14_HUL_051        17          1        +1 Lap
62719  2023_14_MAG_051        18          1        +1 Lap
retired untimed rows that still had a position (set to NaN): 33
- Duplicate groups AFTER: 1
race_id  lap_number  position
2019_17  52          9           2
dtype: int64


---

##### Missing race time

COLUMN : race_time_s

DESCRIPTION : Race time so far: sum of this driver's lap times up to this lap

*Main Dataset*

*The main dataset is in training_laps.parquet and test_laps.parquet. They were generated from data on Formula 1 races 2018-2026: 95 training races, 92 test races. The test races are incomplete : Each test race is cut at a cutoff lap. The cutoff lap was selected when the leader finished a lap between 30 % and 50 % of the race (uniformly at random). It used the column "race_time_s"; you can use this too to generate cuts. Every car shows only the laps it had finished by then, plus the oracle columns. You will be scored on the missing part.*

In [31]:
training[training['race_time_s'].isna()] # 206 missing values

# I realised that for the columns 'driving_time_s' and 'lap_time_s', they had common values multiple times 
# (I only realised this now because since 'race_time_s' is based on the sum of the lap times, I was giving a quick look at the 'lap_time_s' column)
# So I want to check if for all the rows where there's values in both columns ('driving_time_s' and 'lap_time_s'), if the values are the same.
# That's what I'm doing in the next cell

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,False,3.5,997.0,True,False,False,False,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,True,False,False,2.2,996.9,True,False,False,False,True
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,False,2.5,997.2,True,False,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,<NA>,NaN,NaN,True,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,False,0.9,1023.0,True,False,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.0,1023.1,True,False,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,False,0.9,1023.0,True,False,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,False,0.7,1023.0,True,False,True,True,True


In [32]:
# Check if for all the rows where there's values in both columns ('driving_time_s' and 'lap_time_s'), if the values are the same.
both = training[training["lap_time_s"].notna() & training["driving_time_s"].notna()].copy()

both["diff"] = both["driving_time_s"] - both["lap_time_s"]
both["same"] = np.isclose(both["driving_time_s"], both["lap_time_s"], atol=1e-3)

print(len(both))
print(both["same"].value_counts())

# Turns out that for 3637 case, they're not similar

104549
same
True     100912
False      3637
Name: count, dtype: int64


In [33]:
both.loc[~both["same"], ["lap_id", "lap_number", "lap_time_s", "driving_time_s", "diff"]].head(20)

,lap_id,lap_number,lap_time_s,driving_time_s,diff
0,2018_01_ALO_001,1,101.528,101.706,0.178
58,2018_01_BOT_001,1,103.824,104.002,0.178
116,2018_01_ERI_001,1,104.674,104.852,0.178
122,2018_01_GAS_001,1,105.060,105.238,0.178
136,2018_01_GRO_001,1,98.450,98.628,0.178
160,2018_01_HAM_001,1,94.233,94.411,0.178
218,2018_01_HAR_001,1,125.584,125.762,0.178
275,2018_01_HUL_001,1,99.077,99.255,0.178
333,2018_01_LEC_001,1,105.584,105.762,0.178
391,2018_01_MAG_001,1,97.107,97.285,0.178


In [34]:
# It's not going to be possible to compute the 'race_time_s', because - let's say we're on lap 2 - we'll need the 'lap_time_s' of lap 1 AND lap 2 
# (I manually did it using complete rows, that's how I know (Tutorial few lignes below)).
# So if we don't have the 'lap_time_s' of lap 2, we can't compute the 'race_time_s'.

# And we actually don't have the 'lap_time_s' for our missing values here 
# (except 1 row. 'lap_time_s' has 205 missing values and 'race_time_s' has 206 missing values)
# This is the only row where we might compute the 'race_time_s'.
# Other than that we'll have to stick with the missing values

# TUTO on how I did it:
# I used the following rows: 2018_01_ALO_001 & 2018_01_ALO_002
# lap_time_s for 2018_01_ALO_001 : 101.528 s
# lap_time_s for 2018_01_ALO_002 : 91.565 s
# race_time_s for 2018_01_ALO_001: 101.528 s (== lap_time_s for 2018_01_ALO_001 (which makes sense since it's only the first lap))
# race_time_s for 2018_01_ALO_002: 193.093 s  (== 101.528 + 91.565)

In [35]:
# We'll proceed the same way we did with the column 'driving_time_s', encoding

training["untimed_race_time"] = training["race_time_s"].isna()
print("untimed race time:", int(training["untimed_race_time"].sum()))

training[training['race_time_s'].isna()] # It worked


untimed race time: 206


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,2.5,997.2,True,False,True,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,0.9,1023.0,True,False,True,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.0,1023.1,True,False,True,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.9,1023.0,True,False,True,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


---

##### Missing gap to leader

COLUMN : gap_to_leader_s

DESCRIPTION : Seconds behind the leader at the end of this lap

In [36]:
training[training['gap_to_leader_s'].isna()]  # 206 missing values
# First thing we noticed is that those who had a missing gap to leader are driver who didn't finish the race/didn't even start.
# Theory is that the row which has a missing gap to leader is because it's the one where they were forced to retire

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,2.5,997.2,True,False,True,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,0.9,1023.0,True,False,True,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.0,1023.1,True,False,True,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.9,1023.0,True,False,True,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


In [37]:
training.loc[training['gap_to_leader_s'].isna(), 'result_status'].value_counts() 
# According to this, none finished the race

result_status
Retired             92
Collision           42
Accident            14
Engine              11
Did not start        6
Power Unit           5
Brakes               4
Suspension           4
Gearbox              3
Exhaust              3
Hydraulics           3
Spun off             3
+1 Lap               3
Puncture             2
Battery              2
Steering             1
Oil leak             1
Fuel pressure        1
Power loss           1
Collision damage     1
Transmission         1
Wheel nut            1
Driveshaft           1
Front wing           1
Name: count, dtype: int64

In [38]:
training.loc[training['gap_to_leader_s'].isna(), 'position'].value_counts() 
# But according to this, they have a final position after the race, it's not missing

position
9     1
17    1
18    1
Name: count, dtype: Int64

In [39]:
training[(training['gap_to_leader_s'].isna()) & (training['position'] == 1)]
# issue: dans ces 3 cas, there's also another driver with position = 1

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time


In [40]:
# We're looking for cases where more than 1 driver hve position == 1
# It's to see if the problem is only with the previous 3 drivers or not
dups = training[training["position"] == 1].groupby(["race_id", "lap_number"]).size()
dups[dups > 1]

Series([], dtype: int64)

In [41]:
# We're looking for cases where more than 1 driver hve position == 1
# It's to see if the problem is only with the previous 3 drivers or not
dups = training[training["position"] == 1].groupby(["race_id", "lap_number", 'position']).size()
dups[dups > 1]

Series([], dtype: int64)

##### Missing laps down

COLUMN : laps_down

DESCRIPTION : Laps behind the leader (0 = on the lead lap)

In [42]:
training[training['laps_down'].isna()] # 206 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,2.5,997.2,True,False,True,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,0.9,1023.0,True,False,True,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.0,1023.1,True,False,True,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.9,1023.0,True,False,True,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


##### Missing gap to car ahead

Column: gap_to_car_ahead_s

Just realised something.

In the cases where drivers didn't finish the race, it doesn't make sense to fill the missing values with 0 BECAUSE it's also 0 when the car is P1 (they're first, there's no car ahead of them). So if both have a value of 0, I'm scared that the model might fuck up.

In [43]:
training[training['gap_to_car_ahead_s'].isna()] # 5964 missing values
# There's a few reasons why there's missing values in this column:
# - The driver's position is 1. Which means he's leading, which means there's no car ahead. 
#   In this case we could just replace the missing values with 0
# - The driver didn't finish the race fo a certain reason. Which means that at some point during the race, he stoped while the others are still racing. 
#   Which also means that there's also no car ahead of him (makes sense since he's out)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
160,2018_01_HAM_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,False,4.1,997.0,False,False,False,False,False,False
161,2018_01_HAM_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,False,3.8,996.9,False,False,False,False,False,False
162,2018_01_HAM_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAM,44,...,False,False,4.3,997.1,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104643,2026_15_RUS_049,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,<NA>,<NA>,NaN,NaN,False,False,False,False,False,False
104644,2026_15_RUS_050,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,<NA>,<NA>,NaN,NaN,False,False,False,False,False,False
104645,2026_15_RUS_051,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,RUS,63,...,<NA>,<NA>,NaN,NaN,False,False,False,False,False,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


In [44]:
# We'll start by filling the missing values of the drivers who's position == 1
mask = training['gap_to_car_ahead_s'].isna() & (training['position'] == 1)
training.loc[mask, 'gap_to_car_ahead_s'] = 0

In [45]:
# just to make sure that it was filled
x = training[training['position'] == 1]

## Realised something wrong: in 3 cases, the position is 1 but there IS a gap with the car ahead 
## (this gap is also = to the gap to leader)
## This means that either the actual position is 2, or the timing is wrong

In [46]:
training.loc[training['gap_to_car_ahead_s'].isna(), 'position'].value_counts()

position
9     1
2     1
17    1
18    1
Name: count, dtype: Int64

In [47]:
# Check the reasons why the drivers didn't finish
training.loc[training['gap_to_car_ahead_s'].isna(), 'result_status'].value_counts()

## There's 1 case were the result status is 'Finished', which means that the driver actually finished the race.
## What we'll do is that for the rest of the cases, we'll also fill the gap to car ahead with 0.
## For the result_status == Finished case, we'll see if we can figure out the gap. If not, we'll leave it as is (with a missing value).

result_status
Retired             92
Collision           43
Accident            14
Engine              11
Did not start        6
Power Unit           5
Brakes               4
Suspension           4
Gearbox              3
Exhaust              3
Hydraulics           3
Spun off             3
+1 Lap               3
Puncture             2
Collision damage     2
Battery              2
Steering             1
Oil leak             1
Fuel pressure        1
Power loss           1
Transmission         1
Wheel nut            1
Driveshaft           1
Front wing           1
Finished             1
Name: count, dtype: int64

In [48]:
# Getting more information on that case
training[training['gap_to_car_ahead_s'].isna()] 

## So now we know that it's Hamilton during the US GP (19th round) in 2022, during the 40th lap

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,2.5,997.2,True,False,True,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,0.9,1023.0,True,False,True,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.0,1023.1,True,False,True,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.9,1023.0,True,False,True,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


In [49]:
training[(training['race_id'] == '2022_19') & (training['lap_number'] == 40)] 
# According to the data, during that race at lap 40, Hamilton was P2 and and Vettel was P1.
# The thing is, if we look at the 'gap_to_leader_s' column, we see that it's 0 for Hamilton and 0.22 for Vettel. 
# In the 'gap_to_car_ahead_s' column, it's also 0.22 for Vettel.
# In our opinion, this either means that there was a swap/typo in the actual position (Hamilton is actually P1 and Vettel is P2), 
# or when this data was collected, the positions weren't updated

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
51726,2022_19_ALB_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,ALB,23,...,False,False,5.3,990.0,False,False,False,False,False,False
51782,2022_19_ALO_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,ALO,14,...,False,False,5.3,990.0,False,False,False,False,False,False
51855,2022_19_GAS_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,GAS,10,...,False,False,5.3,990.0,False,False,False,False,False,False
51911,2022_19_HAM_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,HAM,44,...,False,False,5.3,990.0,False,False,False,False,False,False
51967,2022_19_LAT_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,LAT,6,...,False,False,5.3,990.0,False,False,False,False,False,False
52023,2022_19_LEC_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,LEC,16,...,False,False,5.3,990.0,False,False,False,False,False,False
52079,2022_19_MAG_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,MAG,20,...,False,False,5.3,990.0,False,False,False,False,False,False
52135,2022_19_MSC_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,MSC,47,...,False,False,5.3,990.0,False,False,False,False,False,False
52191,2022_19_NOR_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,NOR,4,...,False,False,5.3,990.0,False,False,False,False,False,False
52247,2022_19_OCO_040,2022_19,2022,19,United States Grand Prix,Circuit of the Americas,austin,2022-10-23,OCO,31,...,False,False,5.3,990.0,False,False,False,False,False,False


##### Missing gap to car behind

Column: gap_to_car_behind_s

In [50]:
training[training['gap_to_car_behind_s'].isna()]  # 5964 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
121,2018_01_ERI_006,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ERI,9,...,False,False,3.5,997.0,True,False,False,False,True,True
135,2018_01_GAS_014,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,GAS,10,...,False,False,2.2,996.9,True,False,False,False,True,True
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,False,4.1,997.0,False,False,False,False,False,False
219,2018_01_HAR_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,False,3.8,996.9,False,False,False,False,False,False
220,2018_01_HAR_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,False,4.3,997.1,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104701,2026_15_STR_005,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.0,1023.1,False,False,False,False,False,False
104702,2026_15_STR_006,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.5,1023.0,False,False,False,False,False,False
104703,2026_15_STR_007,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,2.2,1022.9,False,False,False,False,False,False
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


In [51]:
training.loc[training['gap_to_car_behind_s'].isna(), 'position'].value_counts()
# What i want to do is figure out what's the last position recorded for lap of each race
# In this case, we'll fill the missing values with 0 because if you're the last car, 
# there's no one behind you

position
20    1488
18    1305
19    1164
17     754
16     379
15     242
14     146
21      84
22      70
13      48
12      26
9       11
6        9
10       9
8        6
5        6
7        6
11       5
4        2
3        1
Name: count, dtype: Int64

In [52]:
# Just to check if in total, we have the 5964 missing values
training.loc[training['gap_to_car_behind_s'].isna(), 'position'].value_counts().sum()
# Nope, only 5791 have a recorded position at the end of the race
# which means that for the 173 other missing values, the driver didn't finish the race
# In this case, i might just leave the missing values as is

np.int64(5761)

The past 4 columns are somehow similar. Here's what I did with Claude

**What I verified** (printed below):
* `gap_to_leader_s` and `laps_down` are missing on exactly the untimed rows (206) — no race time → no gap. (`laps_down` ends with 203 because the 3 `+1 Lap` finishers got their last known value in section 5.)
* `gap_to_car_ahead_s` is missing for **(a)** the car ranked first by time on the lap (nobody ahead) and **(b)** the 206 untimed rows.
  `gap_to_car_behind_s` is missing for the last timed car and the untimed rows. 5758 + 206 = 5964, your number.
* On every timed row `gap_to_car_ahead_s = gap_to_leader_s(this car) − gap_to_leader_s(car in front)`, so these columns carry no independent information.

**Decision (this answers your “0 for the leader vs 0 for retired” worry):** I do **not** fill them with 0.
A gap of 0 would mean “touching the car in front”, which is a real situation, and the leader and a retired car would look identical.
They stay `NaN` here; the two cases can be told apart by `untimed_lap` (retired / not timed) and `position == 1` (leader).
For the model, `model_ready()` at the end turns them into `-1` (an impossible gap).

**2022_19 lap 40 (Hamilton):** by time Hamilton is the leader (gap 0) while `position` says he is 2nd. His `gap_to_car_ahead_s` is empty because, by time, nobody is ahead of him. It is a position/time disagreement, not missing data, so it stays `NaN`.

In [53]:
# I didn't fill any of them

---

##### Missing personal best

Column: personal_best

In [54]:
training[training['personal_best'].isna()] # 137 missing values
# keep in mind that it's a boolean column

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
707,2018_01_SIR_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,SIR,35,...,False,False,2.5,997.2,True,False,True,True,True,True
1039,2018_04_BOT_049,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,BOT,77,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1184,2018_04_GRO_043,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,GRO,8,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
1297,2018_04_HUL_011,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HUL,27,...,False,False,2.8,1022.0,True,False,True,True,True,True
1542,2018_04_RIC_040,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,RIC,3,...,<NA>,<NA>,NaN,NaN,True,False,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104098,2026_15_GAS_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,GAS,10,...,False,False,0.9,1023.0,True,False,True,True,True,True
104440,2026_15_NOR_036,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.0,1023.1,True,False,True,True,True,True
104441,2026_15_NOR_037,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,NOR,1,...,False,False,0.9,1023.0,True,False,True,True,True,True
104704,2026_15_STR_008,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,STR,18,...,False,False,0.7,1023.0,True,False,True,True,True,True


**What happened:** `personal_best` is True when the lap time beats all the driver's *previous* lap times in that race.
I tested that rule on the labelled rows (printed agreement ≈ 99%). 127 of the 137 missing rows have no lap time (a lap without a time cannot be a personal best → `False`);
the other 10 can be computed with the rule.

**Decision:** no lap time → `False`; otherwise apply the rule (lap time < best previous lap time of the same driver in the same race).

In [55]:
d = training.sort_values(KEY + ["lap_number"])
prev_best = d.groupby(KEY)["lap_time_s"].transform(lambda s: s.shift().cummin()).reindex(training.index)
rule = (training["lap_time_s"] < prev_best)               # False when there is no previous lap or no lap time

lab = training["personal_best"].notna() & training["lap_time_s"].notna() & prev_best.notna()
print(f"rule agrees with the existing labels on {lab.sum():,} rows: {(rule[lab] == training.loc[lab, 'personal_best'].astype(bool)).mean():.2%}")

m = training["personal_best"].isna()
print("missing:", int(m.sum()), "| without lap time:", int((m & training['lap_time_s'].isna()).sum()))
training.loc[m, "personal_best"] = rule[m].to_numpy()
print("missing after:", int(training['personal_best'].isna().sum()))

rule agrees with the existing labels on 102,682 rows: 99.08%
missing: 137 | without lap time: 127
missing after: 0


---

##### Missing compound

Column: compound

In [56]:
training[training['compound'].isna()] # There's 27 missing values here

# For all of the 27 missing values, it's the 1st lap of the race 
# (not always the same race, but it's always the 1st lap)!
# What I want to do is have also the 2nd lap for these 27 cases.
# This will allow us to know what compound is each driver is on (unless they pit at that moment (doesn't make sense but maybe))
# Turns out they all have 'official_stop' = True
# I used Claude to do confirm that in the next cell

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time
218,2018_01_HAR_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,HAR,28,...,False,False,4.1,997.0,False,False,False,False,False,False
940,2018_04_ALO_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ALO,14,...,False,False,2.0,1021.8,False,False,False,False,False,False
1040,2018_04_ERI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,ERI,9,...,False,False,2.0,1021.8,False,False,False,False,False,False
1236,2018_04_HAR_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,HAR,28,...,False,False,2.0,1021.8,False,False,False,False,False,False
1349,2018_04_MAG_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,MAG,20,...,False,False,2.0,1021.8,False,False,False,False,False,False
1452,2018_04_RAI_001,2018_04,2018,4,Azerbaijan Grand Prix,Baku City Circuit,baku,2018-04-29,RAI,7,...,False,False,2.0,1021.8,False,False,False,False,False,False
2051,2018_05_HAR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,HAR,28,...,False,True,2.4,1001.5,False,False,False,False,False,False
2504,2018_05_SIR_001,2018_05,2018,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2018-05-13,SIR,35,...,False,True,2.4,1001.5,False,False,False,False,False,False
2920,2018_07_ERI_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,ERI,9,...,False,False,2.7,1017.9,False,False,False,False,False,False
3820,2018_07_VAN_001,2018_07,2018,7,Canadian Grand Prix,Circuit Gilles Villeneuve,montreal,2018-06-10,VAN,2,...,False,False,2.7,1017.9,False,False,False,False,False,False


In [57]:
# the 27 cases (lap 1 rows with missing compound)
missing = training[training["compound"].isna()]

# build a (race_id, driver_code) key for them
keys = pd.MultiIndex.from_frame(missing[["race_id", "driver_code"]])

# keep laps 1 and 2 for those drivers in those races
mask = (
    pd.MultiIndex.from_frame(training[["race_id", "driver_code"]]).isin(keys)
    & training["lap_number"].isin([1, 2])
)

missing_laps_1_2 = (
    training[mask]
    .sort_values(["race_id", "driver_code", "lap_number"])
)

missing_laps_1_2

# Observations: 
# - Each 1st lap has official_stop = True
# - Each 2nd row has stint = 2, which confirms that for those 27 cases, they made a pit stop in the first lap
# And since we don't know what's the compound or what's its actual hardness, we leave it empty
# This applies for compound, coumpound_hardness and hardness scale
# We'll encode those 3 columns

training["unknown_compound"] = training["compound"].isna()
print("unknown compound:", int(training["unknown_compound"].sum()))

unknown compound: 27


##### Missing compound hardness

COLUMN : compound_hardness

DESCRIPTION : Actual dry tyre: C1 (hardest) to C6 (softest); 2018 uses that year's names (e.g. SUPERSOFT).

In [58]:
training[training['compound_hardness'].isna()]

# 'compound_hardness' has 4999 missing values.
# 27 out of the 4999 also have a missig value in the 'compound' column.
# For the rest (the other 4972 values), the 'compound' is either wet or intermediate.
# The hardness in this column only applies on the dry tyres
# And since the missing values in compound are either wet/inters or NA, 
# the compound hardness does not apply
# So as for now we'll leave the column as is & encode it


training["unknown_compound_hardness"] = training["compound_hardness"].isna()
print("unknown compound hardness:", int(training["unknown_compound_hardness"].sum()))

unknown compound hardness: 4999


##### Missing hardness scale

COLUMN : hardness_scale

DESCRIPTION : Which scale compound_hardness uses: 'C1-C6' or '2018 name'.

In [59]:
training[training['hardness_scale'].isna()] # 4999 missing values
#training.loc[training['hardness_scale'].isna(), 'compound'].value_counts()
# What we applied on 'compound_hardness' also applies here

training["unknown_hardness_scale"] = training["hardness_scale"].isna()
print("unknown hardness scale:", int(training["unknown_hardness_scale"].sum()))

unknown hardness scale: 4999


##### Missing tyre life

COLUMN : tyre_life_laps

DESCRIPTION : Laps driven on this set of tyres, including this lap.

In [60]:
training[training['tyre_life_laps'].isna()] # There's 27 missing values here

# At first glance, we can see that for those who the 'tyre_life_laps' is missing, the tyre's compound is also missing 
# (compound, compound_hardness, hardness_scale are all missing).
# Another observation: for all of the 27 missing values, it's the 1st lap of the race 
# (not always the same race, but it's always the 1st lap)!
# This makes things very easy!! 
# Since it's the 1st lap of each of the races, it's also the tyre's 1st lap.
# So all we gotta do is fill the missing values with 1.
# Not true. They might be tires used during the Qualification. In this case, its life isn't 1.
# To have an idea, we'll look at the 2nd lap 

# Extracting the 2nd lap and faire en sorte que we have both rows
missing = training[training["tyre_life_laps"].isna()]

keys = pd.MultiIndex.from_frame(missing[["race_id", "driver_code"]])

mask = (
    pd.MultiIndex.from_frame(training[["race_id", "driver_code"]]).isin(keys)
    & training["lap_number"].isin([1, 2])
)

tyre_laps_1_2 = (
    training[mask]
    .sort_values(["race_id", "driver_code", "lap_number"])
)

cols = ["lap_id", "lap_number", "tyre_life_laps", "fresh_tyre", "stint", "official_stop"]
tyre_laps_1_2[cols]

# Those values CANNOT be filled quelque soit the 'tyre_life_laps' of the 2nd row (even if tyre_life_laps = 2) because the stint IS ALWAYS = 2 in 
# the 2nd lap. Which means they all made a pit stop in the first lap.
# Again, we'll encode it (best not use model_ready() because if we get tyre_life_laps = -1, c'est pas marrant)

training["unknown_tyre_life_laps"] = training["tyre_life_laps"].isna()
print("unknown tyre life:", int(training["unknown_tyre_life_laps"].sum()))

unknown tyre life: 27


---

##### Missing pit lane time

COLUMN : pit_lane_time_s

DESCRIPTION : Time in the pit lane from entry to exit (about 20-30 s).

In [61]:
training[training['pit_lane_time_s'].isna()]

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104752,2026_15_VER_047,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,False,False,False,False,False,False,False,False,False
104753,2026_15_VER_048,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,False,False,False,False,False,False,False,False,False
104754,2026_15_VER_049,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,False,False,False,False,False,False,False,False,False
104755,2026_15_VER_050,2026_15,2026,15,Azerbaijan Grand Prix,Baku City Circuit,baku,2026-09-26,VER,3,...,False,False,False,False,False,False,False,False,False,False


In [62]:
training[training['pit_lane_time_s'].isna()] # 101383 missing values 

# After a quick look, we realised that there's so much missing values because at that moment, 
# the driver didn't pit (no pit, no pit lane time, no value, we end up with missing values (tadaaaa))
# We realised this fact by looking at the official_stop column.
# And to confirm it, we ran the code in the next cell.

training.loc[training['pit_lane_time_s'].isna(), 'official_stop'].value_counts()

# We can see that for ALL the missing values, there was no pit stop. 
# At least we're sure that it's not someone who didn't fill the time for a pit stop procees
# So all we have to do is fill the missing values with 0

training['pit_lane_time_s'] = training['pit_lane_time_s'].fillna(0)

training[training['pit_lane_time_s'].isna()] # they're filled

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps


---

##### Missing track status code

Column: track_status_code

In [63]:
training[training['track_status_code'].isna()] 
# 7 missing values

# W cane try to find out the track status by pulling out the same lap and see what other drivers have
## Ex: the first case of missing track status is Leclerc, during the 1st lap of the Hungarian GP in 2018. 
## What we’ll do is isolate that 1st lap of that race for everyone and look at the track status.
# If it's possible to fill the missing value, fill it.
# And if it's not possible to do this, leave the missing value and encode it
# Apply this to all 7 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps
5826,2018_12_LEC_001,2018_12,2018,12,Hungarian Grand Prix,Hungaroring,hungaroring,2018-07-29,LEC,16,...,True,False,True,True,True,True,False,False,False,False
8147,2018_18_ALO_005,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ALO,14,...,False,True,True,True,True,False,False,False,False,False
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,True,True,True,True,False,False,False,False,False
8748,2018_18_RIC_049,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,RIC,3,...,False,True,True,True,True,True,False,False,False,False
12141,2019_05_NOR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,NOR,4,...,True,False,True,True,True,True,False,False,False,False
12515,2019_05_STR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,STR,18,...,True,False,True,True,True,True,False,False,False,False
21037,2019_19_MAG_053,2019_19,2019,19,United States Grand Prix,Circuit of the Americas,austin,2019-11-03,MAG,20,...,True,False,True,True,True,True,False,False,False,False


In [64]:
# 1st missing value
# Leclerc, Hungarian GP 2018, 1st lap

training[(training['race_id'] == '2018_12') & (training['lap_number'] == 1)] 
# we can see that the track status for this lap is 1 all along (except for Leclerc's missing value)
# so here we'll just fill it with 1

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_12")
training.loc[mask, "track_status_code"] = '1'

In [65]:
# 2nd missing value
# Alonso, US GP 2018, 5th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 5)] 
# Same situation as Leclerc
# We'll proceed the same way

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_18") & (training['lap_number'] == 5)
training.loc[mask, "track_status_code"] = '1'

In [66]:
# 3rd missing value
# GrosJean, US GP 2018, 12th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 12)] 
# It's a bit more complicated here
# For some drivers, track_status_code = 1
# For others, track_status_code = 6>7>1
# For others, track_status_code = 7>1

# So we'll leave it empty for now and deal with the rest of the missing values
# We'll proceed with the encoding at the end (just in case there's the same situation with the other missing values)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps
8159,2018_18_BOT_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,BOT,77,...,False,False,False,False,False,False,False,False,False,False
8215,2018_18_ERI_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,ERI,9,...,False,False,False,False,False,False,False,False,False,False
8270,2018_18_GAS_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GAS,10,...,False,False,False,False,False,False,False,False,False,False
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,False,True,True,True,True,False,False,False,False,False
8328,2018_18_HAM_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HAM,44,...,False,False,False,False,False,False,False,False,False,False
8384,2018_18_HAR_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HAR,28,...,False,False,False,False,False,False,False,False,False,False
8439,2018_18_HUL_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,HUL,27,...,False,False,False,False,False,False,False,False,False,False
8495,2018_18_LEC_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,LEC,16,...,False,False,False,False,False,False,False,False,False,False
8526,2018_18_MAG_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,MAG,20,...,False,False,False,False,False,False,False,False,False,False
8582,2018_18_OCO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,OCO,31,...,False,False,False,False,False,False,False,False,False,False


In [67]:
# 4th missing value
# Ricciardo, US GP 2018, 49th lap

training[(training['race_id'] == '2018_18') & (training['lap_number'] == 49)] 

# Same situation as Leclerc and Alonso
# We'll proceed the same way

mask = training["track_status_code"].isna() & (training["race_id"] == "2018_18") & (training['lap_number'] == 49)
training.loc[mask, "track_status_code"] = '1'

In [68]:
# 5th & 6th missing values
# Norris & Stroll, Spanish GP 2019, 45th lap

training[(training['race_id'] == '2019_05') & (training['lap_number'] == 45)] 

# Also a bit complicated like Grosjean
# The track status code isn't the same for everyone

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_lap,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps
11416,2019_05_ALB_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,ALB,23,...,False,False,False,False,False,False,False,False,False,False
11482,2019_05_BOT_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,BOT,77,...,False,False,False,False,False,False,False,False,False,False
11548,2019_05_GAS_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GAS,10,...,False,False,False,False,False,False,False,False,False,False
11614,2019_05_GIO_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GIO,99,...,False,False,False,False,False,False,False,False,False,False
11680,2019_05_GRO_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,GRO,8,...,False,False,False,False,False,False,False,False,False,False
11746,2019_05_HAM_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,HAM,44,...,False,False,False,False,False,False,False,False,False,False
11812,2019_05_HUL_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,HUL,27,...,False,False,False,False,False,False,False,False,False,False
11878,2019_05_KUB_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,KUB,88,...,False,False,False,False,False,False,False,False,False,False
11943,2019_05_KVY_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,KVY,26,...,False,False,False,False,False,False,False,False,False,False
12009,2019_05_LEC_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,LEC,16,...,False,False,False,False,False,False,False,False,False,False


In [69]:
# 7th missing value
# Magnussen, US GP 2019, 53th lap

training[(training['race_id'] == '2019_19') & (training['lap_number'] == 53)]

# Same situation as Leclerc, Alonso, and Ricciardo
# We'll proceed the same way
mask = training["track_status_code"].isna() & (training["race_id"] == "2019_19") & (training['lap_number'] == 53)
training.loc[mask, "track_status_code"] = '1'


In [70]:
training[training['track_status_code'].isna()]
# Those are the 3 cases where it's a bit tricky
# And because of that, we can't tell what is True and what is False for the following columns:
# green, yellow, safety_car, virtual_sc & red_flag

# We'll encode them (info for the model)

training["untracked_track_status"] = training["track_status_code"].isna()
print("untracked track status:", int(training["untracked_track_status"].sum()))
training[training['track_status_code'].isna()]

untracked track status: 3


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,True,True,True,True,False,False,False,False,False,True
12141,2019_05_NOR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,NOR,4,...,False,True,True,True,True,False,False,False,False,True
12515,2019_05_STR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,STR,18,...,False,True,True,True,True,False,False,False,False,True


##### Missing green (flag I think)

Column: green

In [71]:
training[training['green'].isna()] # 7 missing values

# we can use the track status
mask = training["green"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "green"] = True

training[training['green'].isna()] # 3 missing values left (the ones where we don't have a track status code)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
8316,2018_18_GRO_012,2018_18,2018,18,United States Grand Prix,Circuit of the Americas,austin,2018-10-21,GRO,8,...,True,True,True,True,False,False,False,False,False,True
12141,2019_05_NOR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,NOR,4,...,False,True,True,True,True,False,False,False,False,True
12515,2019_05_STR_045,2019_05,2019,5,Spanish Grand Prix,Circuit de Barcelona-Catalunya,catalunya,2019-05-12,STR,18,...,False,True,True,True,True,False,False,False,False,True


##### Missing yellow (flag)

Column: yellow

In [72]:
training[training['yellow'].isna()]

mask = training["yellow"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "yellow"] = False


##### Missing safety car

Column: safety_car

In [73]:
training[training['safety_car'].isna()]
mask = training["safety_car"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "safety_car"] = False

##### Missing virtual safety car

Column: virtual_sc

In [74]:
training[training['virtual_sc'].isna()]
mask = training["virtual_sc"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "virtual_sc"] = False

##### Missing red flag

Column: red_flag

In [75]:
training[training['red_flag'].isna()]
mask = training["red_flag"].isna() & (training["track_status_code"] == "1")
training.loc[mask, "red_flag"] = False


For those columns between green and red_flag, there's missing values and it's because we have 3 missing values in the track_status_code column. Claude fill them via the **best estimate** (not certainty). Personally, I prefered to keep them empty. We already encoded the track status, so I hope the model learns from it.

---

##### Missing pending penalties

COLUMN : penalty_pending_s

DESCRIPTION : Time penalty pending: seconds announced, not yet served. Can be served at a pit stop; otherwise added to the race time at the end.

In [76]:
training[training['penalty_pending_s'].isna()] # 940 missing values 

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False


In [77]:
missing_penalty = training.loc[training["penalty_pending_s"].isna()]

missing_penalty.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


So all the missing values are in a single race: 2018 Australian GP.

---

##### Missing pending drive through

COLUMN : drive_through_pending

DESCRIPTION : Drive-through penalty pending (must be served in the pit lane within 3 laps)

In [78]:
training[training['drive_through_pending'].isna()] # 940 missing values


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False


In [79]:
missing_drive_through = training.loc[training["drive_through_pending"].isna()]

missing_drive_through.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing as for penalty_pending_s

##### Missing pending stop go

COLUMN : stop_go_pending

DESCRIPTION : Stop-and-go penalty pending (must be served in the pit lane within 3 laps).

In [80]:
training[training['stop_go_pending'].isna()] # 940 missing values


,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False


In [81]:
missing_stop_go = training.loc[training["stop_go_pending"].isna()]

missing_stop_go.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

##### Missing black & orange flag

COLUMN : black_orange_flag

DESCRIPTION : Car shown the black-and-orange flag (damage: must pit) during this lap.

In [82]:
training[training['black_orange_flag'].isna()] # 940 missing values

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,untimed_driving_time,missing_sector1_time,missing_sector2_time,missing_sector3_time,untimed_race_time,unknown_compound,unknown_compound_hardness,unknown_hardness_scale,unknown_tyre_life_laps,untracked_track_status
0,2018_01_ALO_001,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
1,2018_01_ALO_002,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
2,2018_01_ALO_003,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
3,2018_01_ALO_004,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
4,2018_01_ALO_005,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,ALO,14,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
935,2018_01_VET_054,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
936,2018_01_VET_055,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
937,2018_01_VET_056,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False
938,2018_01_VET_057,2018_01,2018,1,Australian Grand Prix,Albert Park Circuit,melbourne,2018-03-25,VET,5,...,False,False,False,False,False,False,False,False,False,False


In [83]:
missing_black_orange_flag = training.loc[training["black_orange_flag"].isna()]

missing_black_orange_flag.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Also Australian GP 2018

A black-and-orange flag is shown during a lap to tell a driver that their car has a mechanical problem or damage and must pit immediately.

2 options:
- The problem could be fixed and the driver could continue racing 
- Retire the car if the fault cannot be safely fixed

We cannot just deduce what happened during the race in this case.

So for now we'll just leave the missing values

##### Missing time penalties announced so far

COLUMN : time_penalties_announced_so_far

DESCRIPTION : Number of time penalties announced for this car so far in the race.

In [84]:
training[training['time_penalties_announced_so_far'].isna()] # 940 missing values

missing_time_penalties_announced_so_far= training.loc[training["time_penalties_announced_so_far"].isna()]

missing_time_penalties_announced_so_far.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

##### Missing black & white flags

COLUMN : black_white_flags_so_far

DESCRIPTION : Black-and-white flags (formal warnings) shown to this car so far.

In [85]:
training[training['black_white_flags_so_far'].isna()] # 940 missing values

missing_black_white_flags = training.loc[training["black_white_flags_so_far"].isna()]

missing_black_white_flags.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing as the black-and-orange flag

##### Missing track limit deletions so far

COLUMN : track_limit_deletions_so_far

DESCRIPTION : Lap times of this car deleted for track limits so far.

In [86]:
training[training['track_limit_deletions_so_far'].isna()] # 940 missing values

missing_track_limit_deletions = training.loc[training["track_limit_deletions_so_far"].isna()]

missing_track_limit_deletions.groupby(
    ["race_id", "race_name"],
    dropna=False
).agg(
    missing_rows=("race_id", "size"),
).sort_values("missing_rows", ascending=False)

,,missing_rows
race_id,race_name,
2018_01,Australian Grand Prix,940


Same thing

Columns: `penalty_pending_s`, `drive_through_pending`, `stop_go_pending`, `black_orange_flag`, `time_penalties_announced_so_far`, `black_white_flags_so_far`, `track_limit_deletions_so_far`.

**What happened:** As we said, the 940 missing rows are *all of one race* - Australia 2018: the whole race-control feed is absent, so it cannot be rebuilt from anything else in the file.

**Decision:** fill with `0` / `False` (it is by far the most common value: e.g. ~98% of laps have no pending penalty) *and* add `race_control_missing = True` so the model knows these zeros are an assumption, not an observation (race_control_missing is an ecoding column).

In [87]:
PEN = ["penalty_pending_s", "drive_through_pending", "stop_go_pending", "black_orange_flag",
       "time_penalties_announced_so_far", "black_white_flags_so_far", "track_limit_deletions_so_far"]
training["race_control_missing"] = training["penalty_pending_s"].isna()
print("races concerned:", training.loc[training['race_control_missing'], 'race_id'].unique())

for col in PEN:
    zero = False if str(training[col].dtype).lower().startswith("bool") else 0
    training[col] = training[col].fillna(zero)
print(training[PEN].isna().sum().to_dict())

races concerned: <ArrowStringArray>
['2018_01']
Length: 1, dtype: str
{'penalty_pending_s': 0, 'drive_through_pending': 0, 'stop_go_pending': 0, 'black_orange_flag': 0, 'time_penalties_announced_so_far': 0, 'black_white_flags_so_far': 0, 'track_limit_deletions_so_far': 0}


---

##### Missing fastest lap L+14

COLUMN : fastest_lap_Lplus14_s

DESCRIPTION : Fastest lap time of any car on race lap L + 14 (L = this row's lap).

In [88]:
training[training['fastest_lap_Lplus14_s'].isna()] # 22536 missing values

training.loc[training['fastest_lap_Lplus14_s'].isna(), 'laps_remaining'].value_counts()


laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing safety car L+14

COLUMN : safety_car_Lplus14

DESCRIPTION : Safety car out on race lap L + 14 (any car's lap).

In [89]:
training[training['safety_car_Lplus14'].isna()] # 22536 missing values

training.loc[training['safety_car_Lplus14'].isna(), 'laps_remaining'].value_counts()


laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing virtual safety car L+14

COLUMN : virtual_sc_Lplus14

DESCRIPTION : Virtual safety car on race lap L + 14.

In [90]:
training[training['virtual_sc_Lplus14'].isna()] # 22536 missing values

training.loc[training['virtual_sc_Lplus14'].isna(), 'laps_remaining'].value_counts()


laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing red flag L+14

COLUMN : red_flag_Lplus14

DESCRIPTION : Red flag on race lap L + 14.

In [91]:
training[training['red_flag_Lplus14'].isna()] # 22536 missing values

training.loc[training['red_flag_Lplus14'].isna(), 'laps_remaining'].value_counts()

laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing rainfall L+14

COLUMN : rainfall_Lplus14

DESCRIPTION : Rain detected by the weather station on race lap L + 14.

In [92]:
training[training['rainfall_Lplus14'].isna()] 

training.loc[training['rainfall_Lplus14'].isna(), 'laps_remaining'].value_counts()

laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing wind speed L+14

COLUMN : wind_speed_Lplus14_ms

DESCRIPTION : Wind speed on race lap L + 14 (median over the cars on that lap).

In [93]:
training[training['wind_speed_Lplus14_ms'].isna()] # 22536 missing values

training.loc[training['wind_speed_Lplus14_ms'].isna(), 'laps_remaining'].value_counts()

laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64

##### Missing pressure L+14

COLUMN : pressure_Lplus14_mbar

DESCRIPTION : Air pressure on race lap L + 14 (median over the cars on that lap).

In [94]:
training[training['pressure_Lplus14_mbar'].isna()] # 22536 missing values

training.loc[training['pressure_Lplus14_mbar'].isna(), 'laps_remaining'].value_counts()

laps_remaining
13    1675
12    1675
11    1671
10    1667
9     1662
8     1660
7     1658
6     1654
5     1647
4     1642
3     1639
2     1610
1     1476
0     1002
14     118
15      65
16      15
Name: count, dtype: Int64


Columns: `fastest_lap_Lplus14_s`, `safety_car_Lplus14`, `virtual_sc_Lplus14`, `red_flag_Lplus14`, `rainfall_Lplus14`, `wind_speed_Lplus14_ms`, `pressure_Lplus14_mbar` (the oracle columns).

**What happened**
* They are always missing **together**, and (99%) exactly when `laps_remaining < 14`: you are asking about lap L+14 and the race is already over.
* Exception: there's 198 rows where races are shorter than scheduled (that's why whe got cases of the remaining laps being >= 14 and yet have missing values in the oracle columns), so the horizon also ran past the real end.
* Every value is the same for all drivers on a given lap — they describe the **race**, not the driver.
* The column is simply the base column 14 laps later: weather / flags at lap L+14, and `fastest_lap_Lplus14_s` = the quickest lap time set by anyone on lap L+14 (my first guess, “best lap so far”, was wrong — only ~19% agreement — so I tested the definition against the data; the corrected rule agrees 99–100% on the non-empty rows, see the printed check).

**Decision:** use the same lookup, but when L+14 is beyond the last lap of the race, use the **last lap of the race** (the latest known state: the weather and the neutralisation status at the chequered flag, and the fastest lap time on that last lap). This is an assumption (“nothing changes after the flag”) but it is the best the file allows, and the lookup rule is validated on the rows that already have a value (printed).

In [95]:
# Proof that some races ended earlier than scheduled
last_lap = training.groupby("race_id")["lap_number"].transform("max")
beyond = training["lap_number"] + 14 > last_lap
na = training["safety_car_Lplus14"].isna()

print((na == beyond).mean())                          # 0.9995
print(training[na & ~beyond].groupby("race_id").size())  # 2018_07: 34, 2019_17: 19

# the 198 rows
x = training[na & (training["laps_remaining"] >= 14)]
x.groupby("race_id").agg(rows=("lap_id", "size"),
                         scheduled=("scheduled_laps", "first"),
                         last_in_data=("lap_number", "max"))

0.9994940672222381
race_id
2018_07    34
2019_17    19
dtype: int64


,rows,scheduled,last_in_data
race_id,,,
2018_07,34,70,56
2019_11,45,67,53
2019_17,19,53,39
2021_01,18,57,43
2022_17,28,61,47
2023_14,38,53,39
2025_11,16,71,57


I still wanted to make sure that those races really ended early. So I asked claude to check again (*my question: so even the race leader in those cases didn't complete everything?*) and here what it said:


Yes, in five of the seven races the leader did not complete the scheduled distance. In the other two he did.

Races ended early (leader's last lap below scheduled_laps):
| Race | Scheduled | Leader’s last lap | Status of the cars on that lap |
| -------- | ------- | ------- | ------- |
| 2021_01 | 57 | 56 | 11 × `Finished` |
| 2025_11 | 71 | 70 | 6 × `Finished` |
| 2023_14 | 53 | 51 | 16 × `Finished`, 2 × `+1 Lap` |
| 2022_17 | 61 | 59 | 12 × `Finished` |
| 2019_11 | 67 | 64 | 13 × `Finished` |

In these races the race was officially completed: the leader and the other lead-lap cars are marked Finished, just on an earlier lap than scheduled. The parquet can’t say why (a red flag, a time limit or something else), and I won’t guess. Among all 95 races, these five are the only ones where Finished drivers stop before scheduled_laps. The other 90 finish on exactly the scheduled lap.

Races where the leader did finish (2018_07 and 2019_17): the leader completes all 70 and 53 laps, with status Finished. In those two races the oracle columns stop 1 to 2 laps early for a reason the data doesn’t show.

In the five shortened races, scheduled_laps is not the real race length, and laps_remaining is calculated from it. The leader’s last lap shows laps_remaining = 1 to 3 instead of 0, and the race never reaches 0. That is where the 14, 15 and 16 in your table come from. You can see it with:

##### Saving the changes in a new parquet

In [96]:
training.to_parquet("cleaned_train_laps.parquet", index=False)
print("saved:", training.shape)

saved: (104757, 84)


In [102]:
# Correlation between the 'next_compound' column and the other columns after dealing with the missing values
# columns to leave out (identifiers / duplicates)
drop = ["lap_id", "date", "race_id", "circuit_id", "result_status"]

cat_cols = [c for c in training.select_dtypes(exclude=["number", "boolean", "bool"]).columns]
num_cols = list(training.select_dtypes(include=["number", "boolean", "bool"]).columns)

data = training.copy()
for c in num_cols: data[c] = data[c].astype("float64")   # fixes the pd.NA errors
for c in cat_cols: data[c] = data[c].astype("object")

data = data.sample(15000, random_state=0)                # speed: the full set is slow
am = association_matrix(data, cat_cols, num_cols)

post_missingvalues_filling = am["next_compound"].drop("next_compound").abs().sort_values(ascending=False)



laps_remaining           0.727841
lap_number               0.721399
race_time_s              0.719492
stint                    0.557815
date                     0.423342
                           ...   
drive_through_pending    0.018719
black_orange_flag        0.018616
lap_time_s               0.018354
stop_go_pending          0.014388
lap_id                        NaN
Name: next_compound, Length: 71, dtype: float64

In [106]:
comparison = pd.concat(
    {
        "pre_missingvalues_filling": pre_missingvalues_filling,
        "post_missingvalues_filling": post_missingvalues_filling,
    },
    axis=1,
)
comparison["same"] = (
    comparison["pre_missingvalues_filling"].eq(comparison["post_missingvalues_filling"])
    | (
        comparison["pre_missingvalues_filling"].isna()
        & comparison["post_missingvalues_filling"].isna()
    )
)
comparison

,pre_missingvalues_filling,post_missingvalues_filling,same
laps_remaining,0.727841,0.727841,True
lap_number,0.721399,0.721399,True
race_time_s,0.719492,0.719492,True
stint,0.557815,0.557815,True
date,0.423342,0.423342,True
...,...,...,...
missing_sector3_time,NaN,0.011585,False
untimed_lap,NaN,NaN,True
untimed_driving_time,NaN,NaN,True
untimed_race_time,NaN,NaN,True


#### 3.2.2 Test Dataset

##### Replacing the missing teams

In [97]:
# Using isna()
test[test['team'].isna()]

# Here, 1 value is missing in the column "team" and it's the following drivers : 
# - Alonso (2018, Race 14, 18th lap)

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
2875,2018_14_ALO_018,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.8,138.0,82.839,False,False,False,True,2.5,995.1


In [98]:
test[(test['race_id'] == '2018_14') & (test['driver_code'] == 'ALO')]

# Just to make sure he was also at McLaren

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar
2866,2018_14_ALO_001,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,True,1.0,225.0,84.582,False,False,False,False,3.1,995.2
2867,2018_14_ALO_002,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,True,0.4,174.0,84.674,False,False,False,False,2.7,995.2
2868,2018_14_ALO_003,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.8,187.0,84.545,False,False,False,False,2.1,995.2
2869,2018_14_ALO_004,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.3,0.0,84.582,False,False,False,False,1.5,995.2
2870,2018_14_ALO_005,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.8,251.0,84.516,False,False,False,False,3.0,995.2
2871,2018_14_ALO_006,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.5,317.0,84.448,False,False,False,False,2.7,995.2
2872,2018_14_ALO_007,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,0.7,197.0,83.611,False,False,False,False,2.6,995.2
2873,2018_14_ALO_008,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,2.6,181.0,83.953,False,False,False,False,0.9,995.1
2874,2018_14_ALO_009,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,2.3,181.0,83.535,False,False,False,False,2.7,995.1
2875,2018_14_ALO_018,2018_14,2018,14,Italian Grand Prix,Autodromo Nazionale Monza,monza,2018-09-02,ALO,14,...,False,1.8,138.0,82.839,False,False,False,True,2.5,995.1


In [99]:
# Fill the missing value with the actual team
test['team'] = test['team'].fillna('McLaren')

# Checking if the missing value was successefully replaced by the actual team name
test[(test['race_id'] == '2018_14') & (test['driver_code'] == 'ALO')] 
test[test['team'].isna()]
## Yes it's good

,lap_id,race_id,year,round,race_name,circuit,circuit_id,date,driver_code,driver_number,...,rainfall,wind_speed_ms,wind_direction_deg,fastest_lap_Lplus14_s,safety_car_Lplus14,virtual_sc_Lplus14,red_flag_Lplus14,rainfall_Lplus14,wind_speed_Lplus14_ms,pressure_Lplus14_mbar


---

## 15. Optional: a model-ready copy

Some models (scikit-learn estimators, neural nets) cannot take `NaN`. Tree models (LightGBM, XGBoost, CatBoost, HistGradientBoosting) can, so for those just use `training` directly.
`model_ready` replaces the remaining numeric gaps with **-1** (no real time, gap or speed is negative, so -1 can never be confused with a real value, and
the leader / retired car can be told apart through `position` and `untimed_lap`).

In [100]:
'''def model_ready(df, sentinel=-1):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if str(s.dtype).lower() == "boolean":
            out[c] = s.fillna(False).astype(bool)
        elif pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            out[c] = s.fillna(sentinel)
    return out

training_model = model_ready(training)
print("NaN left in training_model:", int(training_model.isna().sum().sum()))'''

'def model_ready(df, sentinel=-1):\n    out = df.copy()\n    for c in out.columns:\n        s = out[c]\n        if str(s.dtype).lower() == "boolean":\n            out[c] = s.fillna(False).astype(bool)\n        elif pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):\n            out[c] = s.fillna(sentinel)\n    return out\n\ntraining_model = model_ready(training)\nprint("NaN left in training_model:", int(training_model.isna().sum().sum()))'